# 05 Evaluation
Generates synthetic timing data, compares the supervised and unsupervised behavioral risk signals used by the active workflow, and provides the synthetic-first baseline that should later be aligned against trace-derived windows with `src.behavioral.evaluation`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import hashlib
from statistics import mean, pstdev

from data.generate_synthetic_fsl import generate_timing_bot, generate_timing_normal
from src.behavioral.pow import assess_pow_risk

def build_vector(prefix: str, timings: list[float], entropy_mean: float, entropy_std: float, n_chunks: int) -> dict:
    tau_avg = mean(timings)
    tau_std = pstdev(timings)
    return {
        'tau_avg': tau_avg,
        'tau_std': tau_std,
        'tau_min': min(timings),
        'tau_max': max(timings),
        'interarrival_cv': (tau_std / tau_avg) if tau_avg else 0.0,
        'tau_seq_hash': hashlib.sha256(f'{prefix}:tau'.encode('utf-8')).hexdigest(),
        'entropy_mean': entropy_mean,
        'entropy_std': entropy_std,
        'entropy_min': max(0.0, entropy_mean - entropy_std),
        'entropy_max': min(8.0, entropy_mean + entropy_std),
        'entropy_dist_hash': hashlib.sha256(f'{prefix}:entropy'.encode('utf-8')).hexdigest(),
        'chunk_order_hash': hashlib.sha256(f'{prefix}:order'.encode('utf-8')).hexdigest(),
        'n_chunks': n_chunks,
    }

normal = generate_timing_normal(100)
bot = generate_timing_bot(100)
human_risk = assess_pow_risk(build_vector('human', normal, entropy_mean=7.0, entropy_std=0.2, n_chunks=6))
bot_risk = assess_pow_risk(build_vector('bot', bot, entropy_mean=6.1, entropy_std=1.2, n_chunks=12))
summary = {
    'normal_mean_ms': round(mean(normal), 2),
    'bot_mean_ms': round(mean(bot), 2),
    'human_supervised': human_risk['supervised_prediction'],
    'bot_supervised': bot_risk['supervised_prediction'],
    'human_unsupervised': human_risk['unsupervised_prediction'],
    'bot_unsupervised': bot_risk['unsupervised_prediction'],
    'human_difficulty': human_risk['effective_difficulty'],
    'bot_difficulty': bot_risk['effective_difficulty'],
}
summary


{'normal_mean_ms': 100.19,
 'bot_mean_ms': 2.91,
 'human_supervised': 'human',
 'bot_supervised': 'bot',
 'human_unsupervised': 'outlier',
 'bot_unsupervised': 'outlier',
 'human_difficulty': 16,
 'bot_difficulty': 26}